# Follower (hydrostatic) pressure

A pressure added with `follower=True` in `Shell.add_pressure_load`
stays normal to the deformed mid-surface and acts on its deformed area. In
the undeformed state it equals the dead load returned by
`Shell.calc_fext`. The configuration-dependent part of its force
vector enters `Shell.calc_fint`, and its load stiffness
`Shell.calc_kCfollower`, which is unsymmetric in general, enters
`Shell.calc_kT`, both at the load factor `inc`. The theory, the
truncation of the area vector and the conditions for a symmetric load
stiffness are derived in [Follower pressure loads: theory](../follower_pressure.rst).

## Linear buckling of rings and long cylinders

The linear buckling problem of a hydrostatic pressure is
`lb(kC0, kG + kCfollower)`, where `structsolve.lb` selects the
solvers of unsymmetric matrices when needed. A long cylinder buckles as a
ring, modelled with a quarter of the circumference and symmetry planes:

In [1]:
import numpy as np
from scipy.linalg import eig
from scipy.special import roots_legendre
from structsolve import solve, lb
from panels import modelDB
from panels.shell import Shell

In [2]:
# from tests/tests_shell/test_follower_pressure_validation.py
def ring_shell(model, R, stack, plyts, laminaprops, m=4, n=15, **kwargs):
    s = Shell(model=model, **kwargs)
    s.r = R
    s.a = 0.5*R
    s.b = np.pi/2*R
    s.stack = list(stack)
    s.plyts = list(plyts)
    s.laminaprops = list(laminaprops)
    s.m = m
    s.n = n
    for e in ('x1', 'x2', 'y1', 'y2'):
        for f in 'uvw':
            setattr(s, e + f, 1.)
            setattr(s, e + f + 'r', 1.)
    # symmetry planes at the ends: generalized plane strain
    s.x1u = s.x2u = 0.
    s.x1vr = s.x2vr = 0.
    s.x1wr = s.x2wr = 0.
    # symmetry planes at the straight edges, modes cos(n y/r), n even
    s.y1v = s.y2v = 0.
    s.y1wr = s.y2wr = 0.
    # rotations: phiy free at the ends, phix zero there (symmetry), phiy
    # zero at the straight edges (it varies as sin(n y/r))
    s.x1phiy = s.x2phiy = 1.
    s.x1phix = s.x2phix = 0.
    s.y1phix = s.y2phix = 1.
    s.y1phiy = s.y2phiy = 0.
    s._rebuild()
    return s

In [3]:
# definitions of tests/tests_shell/test_follower_pressure_validation.py used below
def central_load_stiffness(s, p):
    fg = modelDB.db[s.model]['field'].fg
    size = s.get_size()
    g = np.zeros((5, size))
    K = np.zeros((size, size))
    xs, wx = roots_legendre(s.nx)
    ys, wy = roots_legendre(s.ny)
    for xi, wxi in zip(xs, wx):
        for eta, wyi in zip(ys, wy):
            fg(g, (xi + 1)*s.a/2, (eta + 1)*s.b/2, s)
            K -= wxi*wyi*s.a*s.b/4*p/s.r*np.outer(g[1], g[1])
    return K

In [4]:
# from tests/tests_shell/test_follower_pressure_validation.py
def ring_buckling_pressure(s, load='follower', zp=0., return_mode=False):
    s.clear_loads()
    s.add_pressure_load(-1., cte=False, follower=(load == 'follower'), zp=zp)
    k0 = s.calc_kC()
    c0 = solve(k0, s.calc_fext(), silent=True)
    K = s.calc_kG(c=c0)
    if load == 'follower':
        K = K + s.calc_kCfollower()
    elif load == 'central':
        K = K + central_load_stiffness(s, -1.*(1 + zp/s.r))
    eigvals, eigvecs = lb(k0, K, silent=True, num_eigvalues=4)
    if return_mode:
        return eigvals[0], eigvecs[:, 0]
    return eigvals[0]

With the Sanders kinematics the critical pressure of the mode `n = 2` is
`3 D/r**3` for the follower pressure and `4 D/r**3` for the dead load:

In [5]:
# definitions of tests/tests_shell/test_follower_pressure_validation.py used below
E = 70.e9
NU = 0.3
G = E/(2*(1 + NU))
ISO = (E, E, NU, G, G, G)


def ring_closed_form(kinematics, load, n, A, D, R, S=None):
    k = n/R
    kv = 1/R if kinematics == 'sanders' else 0.
    fsdt = S is not None
    nd = 3 if fsdt else 2
    # coefficient vectors of the amplitudes (W, V, P) of each strain, whose
    # integral over the circumference is pi R times the square
    eyy = np.array([1/R, k, 0.])
    if fsdt:
        kyy = np.array([0., kv*k, k])
        gyz = np.array([-k, 0., 1.])
    else:
        kyy = np.array([k**2, kv*k, 0.])
        gyz = np.zeros(3)
    byy = np.array([-k, -kv, 0.])
    H0 = A*np.outer(eyy, eyy) + D*np.outer(kyy, kyy)
    if fsdt:
        H0 += S*np.outer(gyz, gyz)
    H1 = R*np.outer(byy, byy)
    if load == 'follower':
        L = np.array([[1/R, k, 0.], [k, kv, 0.], [0., 0., 0.]])
    elif load == 'dead':
        L = np.zeros((3, 3))
    elif load == 'central':
        L = np.diag([0., 1/R, 0.])
    else:
        raise ValueError(load)
    H1 -= L
    H0 = H0[:nd, :nd]
    H1 = H1[:nd, :nd]
    ps = eig(H0, -H1, right=False)
    ps = ps[np.isfinite(ps)]
    ps = ps[np.abs(ps.imag) <= 1e-10*np.abs(ps)].real
    ps = ps[ps != 0]
    return ps[np.argmin(np.abs(ps))]

In [6]:
# from tests/tests_shell/test_follower_pressure_validation.py
def test_isotropic_ring(model, load):
    R = 1.
    h = 0.01
    s = ring_shell(model, R, [0.], [h], [ISO], fsdt_shear_correction=5/6)
    A22 = s.ABD[1, 1]
    D22 = s.ABD[4, 4]
    S = s.ABD[6, 6] if 'fsdt' in model else None
    kin = 'sanders' if 'sanders' in model else 'donnell'
    ref = -ring_closed_form(kin, load, 2, A22, D22, R, S)
    p_cr = ring_buckling_pressure(s, load)
    # the discretization error of 15 terms along y is below 1e-8
    assert np.isclose(p_cr, ref, rtol=1.e-6), (p_cr, ref)
    inextensional = {('sanders', 'follower'): 3., ('sanders', 'dead'): 4.,
                     ('sanders', 'central'): 4.5,
                     ('donnell', 'follower'): 16/5, ('donnell', 'dead'): 4.,
                     ('donnell', 'central'): 64/15}
    # r/h = 100: extensibility and transverse shear of order (h/r)**2
    assert np.isclose(p_cr*R**3/D22, inextensional[(kin, load)], rtol=1e-3)

In [7]:
for model in ['cylshell_clpt_sanders', 'cylshell_clpt_donnell', 'cylshell_fsdt_sanders', 'cylshell_fsdt_donnell']:
    for load in ['follower', 'dead', 'central']:
        test_isotropic_ring(model=model, load=load)
        print(f"test_isotropic_ring(model={model!r}, load={load!r}): passed")

test_isotropic_ring(model='cylshell_clpt_sanders', load='follower'): passed
test_isotropic_ring(model='cylshell_clpt_sanders', load='dead'): passed
test_isotropic_ring(model='cylshell_clpt_sanders', load='central'): passed
test_isotropic_ring(model='cylshell_clpt_donnell', load='follower'): passed
test_isotropic_ring(model='cylshell_clpt_donnell', load='dead'): passed
test_isotropic_ring(model='cylshell_clpt_donnell', load='central'): passed
test_isotropic_ring(model='cylshell_fsdt_sanders', load='follower'): passed


test_isotropic_ring(model='cylshell_fsdt_sanders', load='dead'): passed
test_isotropic_ring(model='cylshell_fsdt_sanders', load='central'): passed
test_isotropic_ring(model='cylshell_fsdt_donnell', load='follower'): passed
test_isotropic_ring(model='cylshell_fsdt_donnell', load='dead'): passed


test_isotropic_ring(model='cylshell_fsdt_donnell', load='central'): passed


The long sandwich cylinders of Han, Kardomateas and Simitses
[han2004SandwichCylinder] are compared with their shell formulas and, by
running the module, with their elasticity solution, with the pressure on
the outer face given by `zp = h/2`:

In [8]:
# definitions of tests/tests_shell/test_follower_pressure_validation.py used below
HAN2004 = {
    'boron': [(741773, 651125, 899768, 809208, 783556),
              (277305, 253721, 323361, 298839, 287722),
              (70416, 67383, 76087, 73863, 72373),
              (11817, 11717, 12203, 12189, 12106)],
    'graphite': [(720842, 637826, 874654, 790739, 762926),
                 (258549, 238236, 298643, 278606, 268754),
                 (61528, 59207, 65825, 64381, 63244),
                 (9918, 9829, 10168, 10184, 10126)],
    'kevlar': [(605472, 551668, 719856, 673845, 647488),
               (171351, 162433, 188347, 184090, 179624),
               (31418, 30712, 32397, 32633, 32341),
               (4476, 4403, 4470, 4533, 4522)],
}
RATIOS = (15, 30, 60, 120)
INCH = 0.0254
C = 1.0*INCH
GPa = 1.e9
GCORE = 0.0459*GPa/(2*(1 + 0.33))
CORE = (0.0459*GPa, 0.0459*GPa, 0.33, GCORE, GCORE, GCORE)
F = 0.1*INCH
FACES = {
    'boron': (221.0*GPa, 20.7*GPa, 0.23, 5.79*GPa, 5.79*GPa, 3.29*GPa),
    'graphite': (181.0*GPa, 10.3*GPa, 0.28, 7.17*GPa, 7.17*GPa, 5.96*GPa),
    'kevlar': (75.9*GPa, 5.52*GPa, 0.34, 2.28*GPa, 2.28*GPa, 1.89*GPa),
}


def han_shear_stiffness(face, kind, f=F, c=C):
    Gf = FACES[face][4]
    Gc = CORE[4]
    if kind == 'core':
        return c*Gc
    Gbar = (2*f + c)/(2*f/Gf + c/Gc)
    return (2*f + c)*Gbar


def sandwich_ring(face, ratio, model='cylshell_fsdt_sanders', shear=None,
                  f=F, c=C):
    s = ring_shell(model, ratio*(2*f + c), [90., 0., 90.], [f, c, f],
                   [FACES[face], CORE, FACES[face]])
    if shear is not None:
        Abar44 = s.lam.Abar_ts[0, 0]
        s.fsdt_shear_correction = shear/Abar44
        s._rebuild()
        assert np.isclose(s.ABD[6, 6], shear)
    return s

In [9]:
# from tests/tests_shell/test_follower_pressure_validation.py
def test_han2004_shell_formulas(face):
    for i, ratio in enumerate(RATIOS):
        elast, core_only, gbar, s8r, c3d20 = HAN2004[face][i]
        for kind, ref in (('core', core_only), ('Gbar', gbar)):
            S = han_shear_stiffness(face, kind)
            s = sandwich_ring(face, ratio, shear=S)
            p_cr = ring_buckling_pressure(s)
            # the closed form of the same discrete theory
            closed = -ring_closed_form('sanders', 'follower', 2, s.ABD[1, 1],
                                       s.ABD[4, 4], s.r, S)
            assert np.isclose(p_cr, closed, rtol=1e-6)
            # the shell formula of Han et al. uses (EI)eq with Ef, panels the
            # plane-strain D22, E/(1 - nu12 nu21) is 0.5% larger for boron
            # and less for the other faces; the extensibility of the ring is
            # also neglected by the formula
            assert np.isclose(p_cr, ref, rtol=1e-2), (face, ratio, kind,
                                                     p_cr, ref)

In [10]:
for face in ['boron', 'graphite', 'kevlar']:
    test_han2004_shell_formulas(face=face)
    print(f"test_han2004_shell_formulas(face={face!r}): passed")

test_han2004_shell_formulas(face='boron'): passed


test_han2004_shell_formulas(face='graphite'): passed


test_han2004_shell_formulas(face='kevlar'): passed


## Non-linear analyses

The callables of the `Shell` are passed directly to
`structsolve.Analysis`, which passes its load factor to the callables
that accept `inc`, and uses `calc_fext(inc=1., c=c)` as the load vector
of the current configuration in the arc-length methods:

In [11]:
from structsolve import Analysis, solve

In [12]:
from scipy.linalg import eigh

In [13]:
# definitions of tests/tests_shell/test_follower_pressure.py used below
class Recorder(object):
    def __init__(self, s):
        self.s = s
        self.fext = s.calc_fext()
        self.an = None
        self.calls = []

    def calc_fint(self, c, inc=1., silent=True):
        fint = self.s.calc_fint(c, inc=inc)
        R = fint - inc*self.fext
        ref = max(np.linalg.norm(inc*self.fext), np.linalg.norm(fint))
        step = len(self.an.increments)
        self.calls.append((inc, np.linalg.norm(R)/ref, step))
        return fint


def check_quadratic(errors, floor=1.e-12):
    checked = 0
    for e0, e1 in zip(errors[:-1], errors[1:]):
        if floor < e0 < 1.e-2:
            assert e1 <= max(10*e0**1.8, floor), errors
            checked += 1
    orders = [np.log(e2/e1)/np.log(e1/e0)
              for e0, e1, e2 in zip(errors[:-2], errors[1:-1], errors[2:])
              if e2 > floor and e1 < 1e-2]
    return checked, orders


def make_shell(model):
    s = Shell()
    s.model = model
    s.a = 0.3
    s.b = 0.2
    s.r = 0.4 if model.startswith('cylshell') else None
    # unsymmetric and unbalanced, such that A16, A26 and B are non-zero
    s.stack = [30, -45, 0, 90]
    s.plyt = 0.125e-3
    E11 = 142.5e9
    E22 = E11/20
    G12 = 0.5*E22
    s.laminaprop = (E11, E22, 0.25, G12, G12, G12)
    s.m = 5
    s.n = 5
    s.nx = 8
    s.ny = 8
    s.Nxx = s.Nyy = s.Nxy = 0.
    s.Nxx_cte = s.Nyy_cte = s.Nxy_cte = 0.
    for edge in ('x1', 'x2', 'y1', 'y2'):
        for field in 'uvw':
            setattr(s, edge + field, 0. if field == 'w' else 1.)
            setattr(s, edge + field + 'r', 1.)
    # removing the in-plane rigid-body motions
    s.x1u = 0.
    s.y1v = 0.
    s._rebuild()
    return s


def follower_shell(model):
    s = make_shell(model)
    s.add_pressure_load(-8.e3 if model.startswith('cyl') else 1.5e3,
                        follower='quadratic', cte=False)
    s.add_pressure_load(lambda x, y: 2.e3*np.sin(np.pi*x/s.a),
                        y1=0.05, y2=0.15, follower='linear', cte=False)
    return s

In [14]:
# from tests/tests_shell/test_follower_pressure.py
def test_newton_raphson_converges_quadratically(model):
    s = follower_shell(model)
    rec = Recorder(s)
    an = Analysis(s.calc_fext, rec.calc_fint, s.calc_kC, s.calc_kG)
    rec.an = an
    an.initialInc = 0.25
    an.relTOL = 1.e-12
    an.maxNumIter = 20
    an.static(NLgeom=True, silent=True)
    assert np.isclose(an.increments[-1], 1.)
    c = an.cs[-1]
    # deflections well above the thickness, the non-linear terms matter
    h = len(s.stack)*s.plyt
    wmax = np.abs(s.uvw(c, gridx=9, gridy=9)[1]['w']).max()
    assert wmax > 2*h, wmax
    R = s.calc_fext() - s.calc_fint(c)
    assert np.linalg.norm(R) <= 1.e-12*np.linalg.norm(s.calc_fext())
    for step in range(len(an.increments)):
        errors = [e for inc, e, st in rec.calls if st == step]
        checked, orders = check_quadratic(errors)
        assert checked >= 1, (step, errors)
    # the follower terms matter: the dead load gives another solution
    s_dead = make_shell(model)
    for load in s.pressure_loads_inc:
        s_dead.pressure_loads_inc.append(load[:5])
    an_dead = Analysis(s_dead.calc_fext, s_dead.calc_fint, s_dead.calc_kC,
                       s_dead.calc_kG)
    an_dead.initialInc = 0.25
    an_dead.static(NLgeom=True, silent=True)
    assert np.isclose(an_dead.increments[-1], 1.)
    assert (np.linalg.norm(an_dead.cs[-1] - c)
            > 1.e-4*np.linalg.norm(c))

In [15]:
for model in ['plate_clpt_donnell', 'cylshell_clpt_sanders', 'cylshell_fsdt_sanders']:
    test_newton_raphson_converges_quadratically(model=model)
    print(f"test_newton_raphson_converges_quadratically(model={model!r}): passed")

test_newton_raphson_converges_quadratically(model='plate_clpt_donnell'): passed


test_newton_raphson_converges_quadratically(model='cylshell_clpt_sanders'): passed


test_newton_raphson_converges_quadratically(model='cylshell_fsdt_sanders'): passed


## Linear static analysis and kinetic criterion

A geometrically linear structure under a follower load solves the
unsymmetric system `(k0 + kCfollower) c = fext`, which
`structsolve.Analysis` does with `static(NLgeom=False)`, since
`Shell.calc_fext` accepts the configuration `c`:

In [16]:
# from tests/tests_shell/test_follower_pressure.py
def test_linear_static_with_follower_load(model):
    from structsolve import static
    s = make_shell(model)
    s.add_pressure_load(-2.e5, follower=True, cte=False)
    s.add_pressure_load(lambda x, y: 1.e5*(1 + x/s.a), x1=0.1, follower=True,
                        cte=False)
    an = Analysis(s.calc_fext, s.calc_fint, s.calc_kC, s.calc_kG)
    an.relTOL = 1.e-12
    increments, cs = an.static(NLgeom=False, silent=True)
    k0 = s.calc_kC()
    kCf = s.calc_kCfollower()
    fext = s.calc_fext()
    ref = static(k0 + kCf, fext, silent=True)[1][0]
    assert np.allclose(cs[0], ref, rtol=1e-8, atol=1e-10*np.abs(ref).max())
    # the follower terms matter
    c_dead = solve(k0, fext, silent=True)
    assert np.linalg.norm(c_dead - ref) > 1e-4*np.linalg.norm(ref)

In [17]:
for model in ['plate_clpt_donnell', 'cylshell_fsdt_sanders']:
    test_linear_static_with_follower_load(model=model)
    print(f"test_linear_static_with_follower_load(model={model!r}): passed")

test_linear_static_with_follower_load(model='plate_clpt_donnell'): passed
test_linear_static_with_follower_load(model='cylshell_fsdt_sanders'): passed


Non-conservative systems may lose stability by flutter, which only the
kinetic criterion detects: `structsolve.freq` solves the unsymmetric
eigenvalue problem of the tangent stiffness matrix with the load stiffness,
whose eigenvalues become complex at a flutter load. The cantilever cylinder
of [schweizerhof1984Pressure] diverges instead, its lowest frequency
vanishing at the critical load of the static criterion:

In [18]:
# definitions of tests/tests_shell/test_follower_pressure_validation.py used below
SCHWEIZERHOF_E = 2.1e6
SCHWEIZERHOF_NU = 0.3


def schweizerhof_sector(L, R, t, n, bc, model='cylshell_clpt_sanders', m=14,
                        nt=8):
    s = Shell(model=model)
    s.r = R
    s.a = L
    s.b = np.pi*R/n
    s.stack = [0.]
    s.plyt = t
    s.laminaprop = (SCHWEIZERHOF_E, SCHWEIZERHOF_NU)
    s.m = m
    s.n = nt
    for e in ('x1', 'x2', 'y1', 'y2'):
        for f in 'uvw':
            setattr(s, e + f, 1.)
            setattr(s, e + f + 'r', 1.)
    s.y1v = s.y2v = 0.
    s.y1wr = s.y2wr = 0.
    if bc == 'hinged':
        s.x1v = s.x1w = s.x2v = s.x2w = 0.
    elif bc == 'B1':
        s.x1u = s.x1v = s.x1w = 0.
    elif bc == 'B2':
        s.x1v = s.x1w = 0.
    else:
        raise ValueError(bc)
    s._rebuild()
    return s

In [19]:
# from tests/tests_shell/test_follower_pressure_validation.py
def test_kinetic_criterion_cantilever():
    from structsolve import freq
    R = 100.
    s = schweizerhof_sector(6*R, R, 1., 3, 'B1')
    s.rho = 7.8e-9
    s._rebuild()
    s.add_pressure_load(-1., cte=False, follower=True)
    k0 = s.calc_kC()
    c0 = solve(k0, s.calc_fext(), silent=True)
    KL = s.calc_kG(c=c0) + s.calc_kCfollower()
    lam_cr = np.real(lb(k0, KL, silent=True, num_eigvalues=4)[0][0])
    kM = s.calc_kM()
    w2 = []
    for frac in (0., 0.5, 0.9, 1.):
        lambda2 = freq(k0 + frac*lam_cr*KL, kM, silent=True,
                       sparse_solver=False, sort=False)[0]
        finite = np.isfinite(lambda2)
        w2.append(-lambda2[finite][np.argmin(np.abs(lambda2[finite]))])
    w2 = np.array(w2)
    assert np.all(np.abs(w2.imag) <= 1e-8*np.abs(w2[0]))
    assert w2[0].real > w2[1].real > w2[2].real > 0
    assert abs(w2[3]) < 1e-6*abs(w2[0])

In [20]:
test_kinetic_criterion_cantilever()
print('test_kinetic_criterion_cantilever: passed')

test_kinetic_criterion_cantilever: passed


## Validation notebooks

The comparisons with the literature are reproduced, per reference, by the
notebooks of the folder `notebooks`, which use the models and the data of
`tests/tests_shell/test_follower_pressure_validation.py`:

- [han2004_sandwich_rings.ipynb](han2004_sandwich_rings.ipynb): long sandwich cylinders, Tables 2-4 of
  [han2004SandwichCylinder];
- [kardomateas2003_sandwich_rings.ipynb](kardomateas2003_sandwich_rings.ipynb): Tables 2-4 of
  [kardomateas2003Sandwich];
- [kardomateas1993_thick_rings.ipynb](kardomateas1993_thick_rings.ipynb): thick orthotropic and isotropic
  rings, Tables 1 and 2 and Figs. 2 and 5 of [kardomateas1993Orthotropic];
- [schweizerhof1984_pressure_loads.ipynb](schweizerhof1984_pressure_loads.ipynb): Tables 2-4 and Fig. 13 of
  [schweizerhof1984Pressure], rings and cantilever cylinders with an
  unsymmetric load stiffness;
- [nasa_sp8007_external_pressure.ipynb](nasa_sp8007_external_pressure.ipynb): Batdorf's equations for lateral
  and hydrostatic pressure of [nasa2020SP8007].

## References

- **[han2004SandwichCylinder]** J.-H. Han, G. A. Kardomateas and G. J. Simitses, "Elasticity, shell theory and finite element results for the buckling of long sandwich cylindrical shells under external pressure". Composites Part B: Engineering, 35:591-598, 2004. https://doi.org/10.1016/j.compositesb.2003.07.002.
- **[schweizerhof1984Pressure]** K. Schweizerhof and E. Ramm, "Displacement dependent pressure loads in nonlinear finite element analyses". Computers & Structures, 18(6):1099-1114, 1984. https://doi.org/10.1016/0045-7949(84)90154-8.
- **[kardomateas2003Sandwich]** G. A. Kardomateas and G. J. Simitses, "Buckling of long sandwich cylindrical shells under external pressure". In: E. E. Gdoutos and Z. P. Marioli-Riga (eds), Recent Advances in Composite Materials, pp. 291-300, Kluwer Academic Publishers (Springer), Dordrecht, 2003. https://doi.org/10.1007/978-94-017-2852-2_24.
- **[kardomateas1993Orthotropic]** G. A. Kardomateas, "Buckling of thick orthotropic cylindrical shells under external pressure". Journal of Applied Mechanics, 60(1):195-202, 1993. https://doi.org/10.1115/1.2900745.
- **[nasa2020SP8007]** NASA, "Buckling of thin-walled circular cylinders". NASA/SP-8007-2020/REV 2, NASA Langley Research Center, 2020. https://ntrs.nasa.gov/citations/20205011530.